In [0]:
%pip install yfinance
%pip install lxml

In [0]:
import yfinance as yf
from delta.tables import DeltaTable

In [0]:
def rename(col):
    return col.replace("(","").replace("'","").replace(",","").replace(" ","").replace(")","").replace(f"{comp}","")

In [0]:
ticker_rows = spark.table("stocks.reference.tickers").select("ticker").collect()

In [0]:
for row in ticker_rows:
    comp = row.ticker
    data = yf.download(comp, period="3y", auto_adjust=True)
    spark_df = spark.createDataFrame(data.reset_index())
    for col in spark_df.columns:
        spark_df = spark_df.withColumnRenamed(col, rename(col))

    tbl_name = comp.replace("-","_").replace(".","_")
    if not spark.catalog.tableExists(f"stocks.stage.{tbl_name}"):
        spark_df.write.mode("overwrite").format("delta").saveAsTable(f"stocks.stage.{tbl_name}")

    delta_table = DeltaTable.forName(spark, f"stocks.stage.{tbl_name}")

    delta_table.alias("t").merge(
        spark_df.alias("s"),
        "s.Date = t.Date"
    ).whenNotMatchedInsertAll().execute()

In [0]:
# tables = [t.name for t in spark.catalog.listTables("stocks.bronze")]
# for table in tables:
#     spark.sql(f"DROP TABLE stocks.bronze.{table}")